In [42]:
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langgraph.graph import StateGraph , START , END
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from pydantic import BaseModel
from langchain_core.documents import Document

In [31]:
docs = (
    PyPDFLoader("./sample_document_1.pdf").load() +
    PyPDFLoader("./sample_document_2.pdf").load() +
    PyPDFLoader("./sample_document_3.pdf").load()
)

In [32]:
docs

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-02-28T11:31:20+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-28T11:31:20+05:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './sample_document_1.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content="The Evolution of Artificial Intelligence and Machine\n Learning\nArtificial intelligence has undergone remarkable transformations since its inception in the mid-20th\ncentury. The field began with ambitious goals of creating machines that could think and reason like\nhumans, leading to early successes in symbolic reasoning and expert systems. However, these early\napproaches faced significant limitations, particularly in handling uncertainty and learning from\nexperience. The 1980s and 1990s saw the rise of machine learning techniques, which shifted the focus\nfrom hand-coded rules to data-driv

In [33]:
embedding_model = OllamaEmbeddings(model="all-minilm:33m")
llm = OllamaLLM(model = "gemma3:4b")

In [34]:
chunks = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 100).split_documents(docs)

In [40]:
# Create vector store and save to disk for persistence
vector_store = FAISS.from_documents(chunks, embedding_model)
vector_store.save_local("faiss_index")

In [ ]:
# To load the vector store from disk later (skip the above cells):
# vector_store = FAISS.load_local("faiss_index", embedding_model, allow_dangerous_deserialization=True)

In [39]:
retriver = vector_store.as_retriever(search_type = "similarity", search_kwargs = {"k": 4})

In [43]:
class State(BaseModel):
    question: str
    docs: list[Document]
    answer: str

In [ ]:
def retriver(state: State):
    q = state["question"]
    return {"docs": retriver.invoke(q)}

In [ ]:
def generator(state: State):
    q = state["question"]
    relevant_docs = state["docs"]

In [ ]:
graph = StateGraph(State)
graph.add_node("retriver", retriver)
graph.add_nodE("generator", generator)